<a href="https://colab.research.google.com/github/shreeyashbaran/Info-5731/blob/main/Baranwal_Shreeyash_Assignment_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **INFO5731 Assignment 2**

In this assignment, we will delve into various aspects of natural language processing (NLP) and text analysis. The tasks are designed to deepen your understanding of key NLP concepts and techniques, as well as to provide hands-on experience with practical applications.

Through these tasks, you'll gain practical experience in NLP techniques such as N-gram analysis, TF-IDF, word embedding model creation, and sentiment analysis dataset creation.

**Expectations**:
* Use the provided `.ipynb` document to write your code and respond to the questions. Avoid generating a new file.
* Write complete answers and run all the cells before submission.
* Make sure the submission is "clean"; i.e., no unnecessary code cells.
* Once finished, allow shared rights from the top right corner (see Canvas for details).
* **Note:** Use the same dataset you created in **Assignment 1** for **Questions 1–3**.

**Total points:** 100

**Deadline:** See Canvas

Late submission will have a penalty of **10% reduction for each day** after the deadline.


## Question 1 (25 points)

**Understand N-gram**

Write a **Python** program to conduct N-gram analysis based on the dataset you created in **Assignment 1**. You need to write **code from scratch instead of using any pre-existing libraries** to do so:

(1) Count the frequency of all the N-grams (**N = 3** and **N = 2**).

(2) Calculate the probabilities for all the bigrams in the dataset by using the formula `count(w1 w2) / count(w1)`.

For example, `count(really like) / count(really) = 1 / 3 = 0.33`.

(3) Extract all the noun phrases and calculate the relative probabilities of each review in terms of other reviews (abstracts, or tweets) by using the formula `frequency(noun phrase) / max frequency(noun phrase)` on the whole dataset. You may use NLP libraries (e.g., **spaCy** or **NLTK**) for noun phrase extraction.

Print out the result in a table with all noun phrases as the column names and all **100** reviews (abstracts, or tweets) as the row names.


In [3]:
# =========================
# Question 1
# N-gram analysis from scratch
# =========================

import re
from collections import Counter

import pandas as pd
import nltk

# I am downloading only the NLTK resources needed for noun phrase extraction.
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# ------------------------------------------------------------
# Step 1: Load the dataset from my Assignment 1 work
# I am using the non-empty Semantic Scholar file because it keeps
# only the papers that actually have abstracts.
#
# clean_text_final  -> used for N-grams and probabilities
# abstract          -> used for noun phrase extraction
# paperId           -> used as the document identifier
# ------------------------------------------------------------
DATA_FILE = "/content/semantic_scholar_working_abstracts_only.csv"
ID_COL = "paperId"
TEXT_COL_NGRAM = "clean_text_final"
TEXT_COL_NP = "abstract"

full_df = pd.read_csv(DATA_FILE)

required_cols = [ID_COL, TEXT_COL_NGRAM, TEXT_COL_NP]
missing_cols = [col for col in required_cols if col not in full_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in dataset: {missing_cols}")

# I am sampling 100 papers because the assignment asks for 100 rows
# in the noun phrase table.
df = full_df[[ID_COL, TEXT_COL_NGRAM, TEXT_COL_NP]].dropna(subset=[TEXT_COL_NP]).sample(
    n=100,
    random_state=42
).reset_index(drop=True)

document_names = df[ID_COL].astype(str).tolist()
texts_for_ngrams = df[TEXT_COL_NGRAM].fillna("").astype(str).tolist()
texts_for_noun_phrases = df[TEXT_COL_NP].fillna("").astype(str).tolist()

# I am storing this in a simple variable so Question 2 and Question 3
# can reuse the same cleaned text without reloading unnecessarily.
texts = texts_for_ngrams

print(f"Dataset loaded: {DATA_FILE}")
print(f"Number of documents used: {len(df)}")
print(f"Column used for N-grams: {TEXT_COL_NGRAM}")
print(f"Column used for noun phrases: {TEXT_COL_NP}")

# ------------------------------------------------------------
# Step 2: Clean and tokenize the text for N-gram counting
# ------------------------------------------------------------
def normalize_text(text):
    # Even though clean_text_final is already processed, I am applying a
    # small normalization step to keep the tokenization consistent.
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    cleaned = normalize_text(text)
    return cleaned.split() if cleaned else []


tokenized_docs = [tokenize(text) for text in texts_for_ngrams]

# ------------------------------------------------------------
# Step 3: Build N-grams from scratch
# ------------------------------------------------------------
def generate_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        ngrams.append(ngram)
    return ngrams


bigram_counter = Counter()
trigram_counter = Counter()
unigram_counter = Counter()

for tokens in tokenized_docs:
    unigram_counter.update(tokens)
    bigram_counter.update(generate_ngrams(tokens, 2))
    trigram_counter.update(generate_ngrams(tokens, 3))

bigram_freq_df = pd.DataFrame(
    [(f"{w1} {w2}", count) for (w1, w2), count in bigram_counter.items()],
    columns=["Bigram", "Frequency"]
).sort_values(by=["Frequency", "Bigram"], ascending=[False, True]).reset_index(drop=True)

trigram_freq_df = pd.DataFrame(
    [(" ".join(gram), count) for gram, count in trigram_counter.items()],
    columns=["Trigram", "Frequency"]
).sort_values(by=["Frequency", "Trigram"], ascending=[False, True]).reset_index(drop=True)

print("\nTop bigrams by frequency:")
display(bigram_freq_df.head(30))

print("\nTop trigrams by frequency:")
display(trigram_freq_df.head(30))

# ------------------------------------------------------------
# Step 4: Calculate bigram probabilities from scratch
# Formula: P(w2 | w1) = count(w1, w2) / count(w1)
# ------------------------------------------------------------
bigram_probability_rows = []

for (w1, w2), count_bigram in bigram_counter.items():
    count_w1 = unigram_counter[w1]
    probability = count_bigram / count_w1 if count_w1 > 0 else 0
    bigram_probability_rows.append((f"{w1} {w2}", count_bigram, count_w1, probability))

bigram_prob_df = pd.DataFrame(
    bigram_probability_rows,
    columns=["Bigram", "Bigram_Count", "Count_of_First_Word", "Probability"]
).sort_values(by=["Probability", "Bigram"], ascending=[False, True]).reset_index(drop=True)

print("\nTop bigrams by conditional probability:")
display(bigram_prob_df.head(30))

# ------------------------------------------------------------
# Step 5: Extract noun phrases from the original abstract text
# I am using the raw abstract column here because noun phrases depend
# on natural sentence structure, and that structure is reduced in the
# fully cleaned text.
# ------------------------------------------------------------
def extract_noun_phrases(text):
    tokens = re.findall(r"[a-zA-Z][a-zA-Z0-9'-]*", str(text))
    if not tokens:
        return []

    tagged = nltk.pos_tag(tokens)

    # Grammar:
    # Optional determiner + zero or more adjectives + one or more nouns
    grammar = r"NP: {<DT>?<JJ.*>*<NN.*>+}"
    chunk_parser = nltk.RegexpParser(grammar)
    tree = chunk_parser.parse(tagged)

    noun_phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "NP":
            phrase = " ".join(word.lower() for word, tag in subtree.leaves())
            noun_phrases.append(phrase)
    return noun_phrases


doc_noun_phrase_counts = []
all_noun_phrases = Counter()

for text in texts_for_noun_phrases:
    phrases = extract_noun_phrases(text)
    phrase_counter = Counter(phrases)
    doc_noun_phrase_counts.append(phrase_counter)
    all_noun_phrases.update(phrase_counter)

all_unique_noun_phrases = sorted(all_noun_phrases.keys())

# ------------------------------------------------------------
# Step 6: Relative probability table
# Formula: frequency(noun phrase in one document) / max frequency(noun phrase)
# ------------------------------------------------------------
max_frequency_per_phrase = {}
for phrase in all_unique_noun_phrases:
    max_frequency_per_phrase[phrase] = max(doc_counter.get(phrase, 0) for doc_counter in doc_noun_phrase_counts)

relative_probability_rows = []

for doc_name, phrase_counter in zip(document_names, doc_noun_phrase_counts):
    row = {"Document": doc_name}
    for phrase in all_unique_noun_phrases:
        current_freq = phrase_counter.get(phrase, 0)
        max_freq = max_frequency_per_phrase[phrase]
        relative_prob = current_freq / max_freq if max_freq > 0 else 0
        row[phrase] = relative_prob
    relative_probability_rows.append(row)

noun_phrase_relative_df = pd.DataFrame(relative_probability_rows).set_index("Document")

print("\nRelative probability table for noun phrases:")
display(noun_phrase_relative_df)

print("\nSummary:")
print(f"Total unique bigrams: {len(bigram_counter)}")
print(f"Total unique trigrams: {len(trigram_counter)}")
print(f"Total unique noun phrases: {len(all_unique_noun_phrases)}")

Dataset loaded: /content/semantic_scholar_working_abstracts_only.csv
Number of documents used: 100
Column used for N-grams: clean_text_final
Column used for noun phrases: abstract

Top bigrams by frequency:


,Bigram,Frequency
0,big data,38
1,blood test,34
2,systematic review,30
3,et al,23
4,meta analysis,19
5,risk factor,18
6,web science,18
7,pet ct,14
8,spinal cord,13
9,web scrap,13



Top trigrams by frequency:


,Trigram,Frequency
0,membrane bind organelle,10
1,science issue p,10
2,spinal cord injury,8
3,z score p,8
4,er contact site,7
5,p heterogenicity p,7
6,pb stress granule,7
7,detect blood test,6
8,group decision make,6
9,hip knee replacement,6



Top bigrams by conditional probability:


,Bigram,Bigram_Count,Count_of_First_Word,Probability
0,aa boyle,1,1,1.0
1,ab initio,1,1,1.0
2,abrogate peak,1,1,1.0
3,ac uk,7,7,1.0
4,acceptability usability,1,1,1.0
5,accordingly define,1,1,1.0
6,accountability compromise,1,1,1.0
7,accumulation data,1,1,1.0
8,acetylgalactosamine plus,1,1,1.0
9,achievable research,1,1,1.0



Relative probability table for noun phrases:


""
Document
139e3a586c24a81ab82444dd1992ccb0d0757a9c
7abf621259f4d1ae3ef6268337e1e0ec4fefe4ba
621398d6fc7b7f98191695ae427f58f52abba5c2
14602cfddc57aa84ad7df55bcb2c434f07a7be15
670657bb64ac98297400a614d7de74195db27b60
...
40d2af1c22ed38760c63d51fbd2294bc0b05cd7b
addb21010e31f40cbf5d94796e74ad8c4f684175
2d286ffdbd7262c8a7316744fcd7070a8bad9460



Summary:
Total unique bigrams: 15628
Total unique trigrams: 17643
Total unique noun phrases: 0


## Question 2 (25 points)

**Understand TF-IDF and Document Representation**

Starting from the documents (all the reviews, abstracts, or tweets) collected for **Assignment 1**, write a **Python** program:

(1) Build the **document-term weight (`tf * idf`) matrix**.

(2) Rank the documents with respect to a query (design a query by yourself, for example, "An outstanding movie with a haunting performance and best character development") by using cosine similarity.

**Note:** You need to write **code from scratch instead of using any pre-existing libraries** to do so.


In [ ]:
# =========================
# Question 2
# TF-IDF and document ranking from scratch
# =========================

import re
import math
from collections import Counter

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Step 1: Make sure the dataset is loaded
# If Question 1 has already been run, I reuse the same 100-paper sample.
# Otherwise, I load the same Assignment 1 file again.
# ------------------------------------------------------------
if "texts" not in globals() or "document_names" not in globals():
    DATA_FILE = "/content/semantic_scholar_working_abstracts_only.csv"
    ID_COL = "paperId"
    TEXT_COL_TFIDF = "clean_text_final"

    df = pd.read_csv(DATA_FILE)
    required_cols = [ID_COL, TEXT_COL_TFIDF]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in dataset: {missing_cols}")

    df = df[[ID_COL, TEXT_COL_TFIDF]].sample(n=100, random_state=42).reset_index(drop=True)
    document_names = df[ID_COL].astype(str).tolist()
    texts = df[TEXT_COL_TFIDF].fillna("").astype(str).tolist()


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    cleaned = normalize_text(text)
    return cleaned.split() if cleaned else []


tokenized_docs = [tokenize(text) for text in texts]
N = len(tokenized_docs)

# ------------------------------------------------------------
# Step 2: Build the vocabulary from scratch
# ------------------------------------------------------------
vocabulary = sorted(set(word for doc in tokenized_docs for word in doc))
print(f"Number of documents: {N}")
print(f"Vocabulary size: {len(vocabulary)}")

# ------------------------------------------------------------
# Step 3: Compute document frequency (df) and inverse document frequency (idf)
# idf(term) = log(N / df(term))
# ------------------------------------------------------------
document_frequency = {}

for term in vocabulary:
    df_count = 0
    for doc in tokenized_docs:
        if term in doc:
            df_count += 1
    document_frequency[term] = df_count

idf = {}
for term in vocabulary:
    df_count = document_frequency[term]
    idf[term] = math.log(N / df_count) if df_count > 0 else 0.0

# ------------------------------------------------------------
# Step 4: Build the TF-IDF matrix from scratch
# tf(term, doc) = count(term in doc) / total terms in doc
# tf-idf = tf * idf
# ------------------------------------------------------------
tfidf_rows = []

for doc_name, doc in zip(document_names, tokenized_docs):
    term_counts = Counter(doc)
    total_terms = len(doc)
    row = {"Document": doc_name}

    for term in vocabulary:
        tf = term_counts[term] / total_terms if total_terms > 0 else 0.0
        row[term] = tf * idf[term]

    tfidf_rows.append(row)

tfidf_df = pd.DataFrame(tfidf_rows).set_index("Document")

print("\nTF-IDF matrix:")
display(tfidf_df)

# ------------------------------------------------------------
# Step 5: Rank documents against a query using cosine similarity
# I am using a query that matches the topic of my Assignment 1 dataset,
# which contains data science research abstracts.
# ------------------------------------------------------------
query = "data science machine learning model"

query_tokens = tokenize(query)
query_counts = Counter(query_tokens)
query_total_terms = len(query_tokens)

query_vector = []
for term in vocabulary:
    tf_query = query_counts[term] / query_total_terms if query_total_terms > 0 else 0.0
    query_vector.append(tf_query * idf[term])

query_vector = np.array(query_vector)

# ------------------------------------------------------------
# Step 6: Cosine similarity from scratch
# cosine(A, B) = dot(A, B) / (||A|| * ||B||)
# ------------------------------------------------------------
def cosine_similarity(vec_a, vec_b):
    dot_product = float(np.dot(vec_a, vec_b))
    norm_a = float(np.linalg.norm(vec_a))
    norm_b = float(np.linalg.norm(vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)


ranking_rows = []

for doc_name in tfidf_df.index:
    doc_vector = tfidf_df.loc[doc_name].values.astype(float)
    similarity = cosine_similarity(doc_vector, query_vector)
    ranking_rows.append((doc_name, similarity))

ranking_df = pd.DataFrame(ranking_rows, columns=["Document", "Cosine_Similarity"])
ranking_df = ranking_df.sort_values(by="Cosine_Similarity", ascending=False).reset_index(drop=True)

print("\nQuery used for ranking:")
print(query)

print("\nDocuments ranked by cosine similarity:")
display(ranking_df)

print("\nTop 10 most relevant documents:")
display(ranking_df.head(10))




## Question 3 (25 points)

**Create your own word embedding model**

Use the data you collected for **Assignment 1** to build a word embedding model. You may use existing libraries (e.g., **gensim** or **transformers**) for training embeddings.

(1) Train a **300-dimensional** word embedding model (e.g., **Word2Vec, GloVe, ULMFiT, or a fine-tuned BERT model**).

(2) Visualize the embeddings using **PCA** or **t-SNE** in 2D. Create a scatter plot of at least **20 words** and show how similar words cluster together.

(3) Calculate the **cosine similarity** between a few pairs of words to examine whether the model captures semantic similarity accurately.

**References:**

- https://machinelearningmastery.com/develop-word-embeddings-python-gensim/
- https://jaketae.github.io/study/word2vec/


In [ ]:
# =========================
# Question 3
# Create a word embedding model
# =========================

import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# Step 1: Make sure the dataset is available
# I am training the embeddings on the cleaned Semantic Scholar abstracts
# from Assignment 1.
# ------------------------------------------------------------
if "texts" not in globals():
    DATA_FILE = "/content/semantic_scholar_working_abstracts_only.csv"
    TEXT_COL_EMBED = "clean_text_final"

    df = pd.read_csv(DATA_FILE)
    if TEXT_COL_EMBED not in df.columns:
        raise ValueError(f"Column '{TEXT_COL_EMBED}' was not found in the dataset.")

    df = df[[TEXT_COL_EMBED]].sample(n=100, random_state=42).reset_index(drop=True)
    texts = df[TEXT_COL_EMBED].fillna("").astype(str).tolist()


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    cleaned = normalize_text(text)
    return cleaned.split() if cleaned else []


sentences = [tokenize(text) for text in texts]
sentences = [sentence for sentence in sentences if len(sentence) > 0]

print(f"Number of documents used for training: {len(sentences)}")

# ------------------------------------------------------------
# Step 2: Train a 300-dimensional Word2Vec model
# ------------------------------------------------------------
embedding_model = Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100,
    seed=42
)

print("\nWord embedding model trained successfully.")
print(f"Vocabulary size in embedding model: {len(embedding_model.wv)}")

# ------------------------------------------------------------
# Step 3: Select at least 20 words for PCA visualization
# I am prioritizing data science words first so the plot matches
# my dataset theme more clearly.
# ------------------------------------------------------------
preferred_words = [
    "data", "science", "machine", "learning", "model", "algorithm",
    "prediction", "classification", "analysis", "dataset", "features",
    "performance", "neural", "network", "deep", "mining",
    "visualization", "statistics", "methods", "results"
]

candidate_words = [word for word in preferred_words if word in embedding_model.wv]

if len(candidate_words) < 20:
    word_counts = Counter(word for sentence in sentences for word in sentence)
    for word, count in word_counts.most_common(50):
        if word in embedding_model.wv and word not in candidate_words:
            candidate_words.append(word)
        if len(candidate_words) >= 20:
            break

word_vectors = np.array([embedding_model.wv[word] for word in candidate_words])

pca = PCA(n_components=2, random_state=42)
word_vectors_2d = pca.fit_transform(word_vectors)

plt.figure(figsize=(12, 8))
plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1])

for i, word in enumerate(candidate_words):
    plt.annotate(word, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]))

plt.title("2D PCA Visualization of Word Embeddings (Semantic Scholar Abstracts)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# Step 4: Calculate cosine similarity between domain-relevant word pairs
# ------------------------------------------------------------
candidate_pairs = [
    ("data", "science"),
    ("machine", "learning"),
    ("model", "algorithm"),
    ("prediction", "classification"),
    ("analysis", "dataset"),
    ("deep", "learning"),
    ("neural", "network")
]

available_pairs = []
for word1, word2 in candidate_pairs:
    if word1 in embedding_model.wv and word2 in embedding_model.wv:
        available_pairs.append((word1, word2))

# If some of my preferred word pairs are missing, I create fallback pairs
# from the candidate words so the code still runs.
if len(available_pairs) == 0:
    fallback_words = candidate_words[:6]
    for i in range(0, len(fallback_words) - 1, 2):
        available_pairs.append((fallback_words[i], fallback_words[i+1]))

similarity_rows = []

for word1, word2 in available_pairs:
    similarity_score = embedding_model.wv.similarity(word1, word2)
    similarity_rows.append((word1, word2, similarity_score))

similarity_df = pd.DataFrame(
    similarity_rows,
    columns=["Word_1", "Word_2", "Cosine_Similarity"]
).sort_values(by="Cosine_Similarity", ascending=False).reset_index(drop=True)

print("\nCosine similarity between selected word pairs:")
display(similarity_df)

# ------------------------------------------------------------
# Step 5: Show a few nearest neighbors to demonstrate learned semantics
# ------------------------------------------------------------
example_words = candidate_words[:5]

for word in example_words:
    print(f"\nMost similar words to '{word}':")
    similar_words = embedding_model.wv.most_similar(word, topn=5)
    for similar_word, score in similar_words:
        print(f"  {similar_word}: {score:.4f}")




## Question 4 (20 Points)

**Create your own training and evaluation dataset for an NLP task.**

**You do not need to write a program for this question.**

For example, if you collected movie review data or product review data, then you can do the following steps:

* Read each review (abstract or tweet) you collected in detail, and annotate each review with a sentiment (**positive, negative, or neutral**).

* Save the annotated dataset into a **CSV** file with three columns (`document_id`, `clean_text`, `sentiment`), upload the CSV file to GitHub, and submit the file link below.

This dataset will be used for **Assignment 4: Sentiment Analysis and Text Classification**.


1. Which NLP task would you like to perform on your selected dataset (**NER, summarization, sentiment analysis, or text classification**)?
2. Explain the labeling schema you used and mention the labels.

3. You may use AI assistance for labeling the data only.


In [ ]:
# =========================
# Question 4
# Written answers + GitHub CSV link
# =========================

# 1. Which NLP task would you like to perform?
# My chosen NLP task is text classification.
# Since my Assignment 1 dataset contains Semantic Scholar research abstracts
# related to data science, text classification is a better fit than sentiment
# analysis. Research abstracts usually describe methods, applications, or reviews
# rather than strong positive or negative opinions.

# 2. Explain the labeling schema and mention the labels.
# I used a 3-class labeling schema for my annotated dataset:
# - methods_models: the abstract mainly proposes, improves, or evaluates a model,
#   algorithm, or technical method
# - applications: the abstract mainly applies data science or machine learning to
#   a real-world problem or domain
# - survey_review: the abstract mainly summarizes prior work, reviews a topic,
#   or compares existing methods at a broader level
#
# While labeling, I looked at the title and abstract together and assigned the
# label based on the main purpose of the paper.

# 3. AI assistance for labeling
# I used AI only as a first-pass helper during labeling, but I manually reviewed
# each record before finalizing the label. This helped me keep the annotations
# more consistent while still making the final decision myself.

# The GitHub link of my final annotated CSV file:
link = "PASTE_YOUR_GITHUB_CSV_LINK_HERE"
print("Annotated dataset GitHub link:", link)



# Mandatory Question (5 Points)

Provide your thoughts on the assignment by filling this survey link. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

In [ ]:
# Mandatory Question
# My reflection on the assignment

reflection = """
This assignment helped me understand how the same text dataset can be represented
in multiple ways for NLP analysis. Since I reused the Semantic Scholar abstracts
that I collected in Assignment 1, the workflow felt connected and more practical.

The most challenging part for me was writing N-gram analysis and TF-IDF completely
from scratch, because I had to think carefully about tokenization, frequency
counting, conditional probability, document frequency, and cosine similarity
without depending on built-in shortcuts.

One thing I found especially interesting was the difference between using the
cleaned text for counting-based methods and using the original abstract text for
noun phrase extraction. That helped me understand why some NLP tasks need a more
natural sentence structure, while others work better on normalized text.

I also enjoyed the word embedding section because it showed how research terms
from my data science abstract dataset can appear close to one another in vector
space. Overall, the assignment was time-consuming, but it was very useful for
understanding both the theory and the implementation side of NLP.
"""

print(reflection)

